In [3]:
# Cell 1 — imports
import pandas as pd
import numpy as np

# Cell 2 — load data
df = pd.read_csv('../data/bengaluru_house_prices.csv')
df.shape

(13320, 9)

In [4]:
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [5]:
df.dtypes

area_type           str
availability        str
location            str
size                str
society             str
total_sqft          str
bath            float64
balcony         float64
price           float64
dtype: object

In [6]:
df.isnull().sum()

area_type          0
availability       0
location           1
size              16
society         5502
total_sqft         0
bath              73
balcony          609
price              0
dtype: int64

In [7]:
# Cell 6 — look at total_sqft for anything that isn't a plain number
def is_plain_number(x):
    try:
        float(x)
        return True
    except ValueError:
        return False

weird_sqft = df[~df['total_sqft'].apply(is_plain_number)]
print(len(weird_sqft))
weird_sqft['total_sqft'].unique()

247


<StringArray>
[    '2100 - 2850',     '3010 - 3410',     '2957 - 3450',     '3067 - 8156',
     '1042 - 1105',     '1145 - 1340',     '1015 - 1540',     '1520 - 1740',
  '34.46Sq. Meter',     '1195 - 1440',
 ...
     '1400 - 1421',     '4000 - 4450', '142.84Sq. Meter',    '300Sq. Yards',
     '2204 - 2362',     '1437 - 1629',      '850 - 1060',     '1200 - 1470',
     '1020 - 1130',     '1133 - 1384']
Length: 222, dtype: str

In [8]:
# Cell 7 — look at size values
df['size'].value_counts()

size
2 BHK         5199
3 BHK         4310
4 Bedroom      826
4 BHK          591
3 Bedroom      547
1 BHK          538
2 Bedroom      329
5 Bedroom      297
6 Bedroom      191
1 Bedroom      105
8 Bedroom       84
7 Bedroom       83
5 BHK           59
9 Bedroom       46
6 BHK           30
7 BHK           17
1 RK            13
10 Bedroom      12
9 BHK            8
8 BHK            5
11 BHK           2
11 Bedroom       2
10 BHK           2
27 BHK           1
19 BHK           1
16 BHK           1
43 Bedroom       1
14 BHK           1
12 Bedroom       1
13 BHK           1
18 Bedroom       1
Name: count, dtype: int64

In [9]:
# Cell 8 — isolate the non-range weird values and see every unique unit suffix
import re

non_range = weird_sqft[~weird_sqft['total_sqft'].str.contains('-')]
non_range['total_sqft'].unique()

<StringArray>
[ '34.46Sq. Meter',       '4125Perch',   '1000Sq. Meter',   '1100Sq. Yards',
       '5.31Acres',         '30Acres',    '716Sq. Meter',   '1500Sq. Meter',
 '142.61Sq. Meter',   '1574Sq. Yards', '361.33Sq. Yards',    '117Sq. Yards',
   '3040Sq. Meter',    '500Sq. Yards',    '167Sq. Meter',    '315Sq. Yards',
          '3Cents', '188.89Sq. Yards',    '204Sq. Meter',     '45Sq. Yards',
  '133.3Sq. Yards',  '78.03Sq. Meter',    '122Sq. Yards',  '84.53Sq. Meter',
       '2.09Acres',        '24Guntha',    '697Sq. Meter',       '1500Cents',
    '132Sq. Yards',          '2Acres',   '1100Sq. Meter',         '15Acres',
       '1.26Acres', '151.11Sq. Yards',        '1Grounds',   '2940Sq. Yards',
  '45.06Sq. Meter',       '1.25Acres',  '86.72Sq. Meter',        '38Guntha',
          '6Acres',    '120Sq. Yards',     '24Sq. Meter', '142.84Sq. Meter',
    '300Sq. Yards']
Length: 45, dtype: str

In [11]:
# Cell 9 — build the full parse_sqft function

UNIT_TO_SQFT = {
    'Sq. Meter': 10.7639,
    'Sq. Yards': 9,
    'Acres': 43560,
    'Guntha': 1089,
    'Perch': 272.25,
    'Cents': 435.6,
    'Grounds': 2400,
}

def parse_sqft(x):
    x = str(x).strip()

    # Range like "2100 - 2850" -> mean of the two ends
    if '-' in x:
        try:
            lo, hi = x.split('-')
            return (float(lo.strip()) + float(hi.strip())) / 2
        except ValueError:
            return None

    # Value with a unit suffix, e.g. "34.46Sq. Meter", "3Cents"
    match = re.match(r'^([\d.]+)([A-Za-z. ]+)$', x)
    if match:
        number, unit = match.groups()
        unit = unit.strip()
        if unit in UNIT_TO_SQFT:
            return float(number) * UNIT_TO_SQFT[unit]
        return None  # unrecognized unit — surface this, don't silently guess

    # Plain number
    try:
        return float(x)
    except ValueError:
        return None

In [12]:
# Cell 10 — apply it and check nothing fell through as None
df['total_sqft_clean'] = df['total_sqft'].apply(parse_sqft)
df['total_sqft_clean'].isnull().sum()

np.int64(0)

In [13]:
# Cell 11 — only if Cell 10 showed nulls > 0
df[df['total_sqft_clean'].isnull()]['total_sqft'].unique()

<StringArray>
[]
Length: 0, dtype: str

In [14]:
# Cell 12 — extract BHK count
df['bhk'] = df['size'].str.extract(r'(\d+)').astype(float)
df['bhk'].value_counts(dropna=False)

bhk
2.0     5528
3.0     4857
4.0     1417
1.0      656
5.0      356
6.0      221
7.0      100
8.0       89
9.0       54
NaN       16
10.0      14
11.0       4
27.0       1
19.0       1
16.0       1
43.0       1
14.0       1
12.0       1
13.0       1
18.0       1
Name: count, dtype: int64

In [15]:
# Cell 13 — drop the row missing `location`, and rows missing `size`
df = df.dropna(subset=['location', 'size'])
df.shape

(13303, 11)

In [16]:
# Cell 14 — drop `society` entirely (41% missing, too sparse to impute or encode)
df = df.drop(columns=['society'])
df.columns

Index(['area_type', 'availability', 'location', 'size', 'total_sqft', 'bath',
       'balcony', 'price', 'total_sqft_clean', 'bhk'],
      dtype='str')

In [17]:
# Cell 15 — median-impute bath and balcony (low missing rate, safe to fill)
df['bath'] = df['bath'].fillna(df['bath'].median())
df['balcony'] = df['balcony'].fillna(df['balcony'].median())
df.isnull().sum()

area_type           0
availability        0
location            0
size                0
total_sqft          0
bath                0
balcony             0
price               0
total_sqft_clean    0
bhk                 0
dtype: int64

In [18]:
# Cell 16 — drop the raw columns now that we have clean versions
df = df.drop(columns=['size', 'total_sqft'])
df = df.rename(columns={'total_sqft_clean': 'total_sqft'})
df.head()

,area_type,availability,location,bath,balcony,price,total_sqft,bhk
0,Super built-up Area,19-Dec,Electronic City Phase II,2.0,1.0,39.07,1056.0,2.0
1,Plot Area,Ready To Move,Chikka Tirupathi,5.0,3.0,120.00,2600.0,4.0
2,Built-up Area,Ready To Move,Uttarahalli,2.0,3.0,62.00,1440.0,3.0
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3.0,1.0,95.00,1521.0,3.0
4,Super built-up Area,Ready To Move,Kothanur,2.0,1.0,51.00,1200.0,2.0


In [19]:
# Cell 17 — save the cleaned data for step 3
df.to_csv('../data/cleaned_house_prices.csv', index=False)